In [5]:
import sys
sys.path.append('../')

import math
from itertools import product
import pandas as pd
import numpy as np
from src.models import BM25
from src.PreProcessingPipeline import BM25_PreProcess


In [12]:
import sys
sys.path.append("../")

import math
import importlib
import numpy as np
import pandas as pd
from itertools import product

import src.query_data
importlib.reload(src.query_data)

from src.query_data import corpus_df, corpus, processed_corpus, query, query_prepro
from src.models import BM25

Columns: ['category', 'filename', 'title', 'content']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\charl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\charl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [13]:
assert len(corpus) == len(processed_corpus), "Corpus and processed_corpus lengths do not match"
assert len(query) == len(query_prepro), "Query and query_prepro lengths do not match"
assert "category" in corpus_df.columns, "corpus_df must contain a category column"
assert "category" in query.columns, "query must contain a category column"

print("All data checks passed.")
print("Documents:", len(corpus))
print("Queries:", len(query))

All data checks passed.
Documents: 2225
Queries: 25


In [14]:
def average_precision(relevance):
    """
    relevance: list of 0/1 values in ranked order
    """
    num_rel = sum(relevance)
    if num_rel == 0:
        return 0.0

    precisions = []
    rel_found = 0

    for i, rel in enumerate(relevance, start=1):
        if rel == 1:
            rel_found += 1
            precisions.append(rel_found / i)

    return sum(precisions) / num_rel


def precision_at_k(relevance, k=10):
    if len(relevance) == 0:
        return 0.0
    return sum(relevance[:k]) / k


def dcg_at_k(relevance, k=10):
    relevance = relevance[:k]
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))


def ndcg_at_k(relevance, k=10):
    dcg = dcg_at_k(relevance, k)
    ideal = sorted(relevance, reverse=True)
    idcg = dcg_at_k(ideal, k)
    return dcg / idcg if idcg > 0 else 0.0

In [20]:
def evaluate_bm25_model(bm25, query_df, query_prepro_list, ndcg_k=10, max_queries=None):
    ap_scores = []
    p10_scores = []
    ndcg_scores = []
    per_query_results = []

    n_queries = len(query_df) if max_queries is None else min(max_queries, len(query_df))

    for i in range(n_queries):

        preprocessed_query = query_prepro_list[i]
        relevant_category = query_df.iloc[i]["category"]
        raw_query = query_df.iloc[i]["query"]

        ranked_results = bm25.rank(preprocessed_query)

        if not isinstance(ranked_results, pd.DataFrame):
            raise TypeError("bm25.rank(preprocessed_query) must return a pandas DataFrame")

        if "category" not in ranked_results.columns:
            raise KeyError("The ranked results must contain a 'category' column")

        relevance = ranked_results["category"].eq(relevant_category).astype(int).tolist()

        ap = average_precision(relevance)
        p10 = precision_at_k(relevance, k=10)
        ndcg = ndcg_at_k(relevance, k=ndcg_k)

        ap_scores.append(ap)
        p10_scores.append(p10)
        ndcg_scores.append(ndcg)

        per_query_results.append({
            "query": raw_query,
            "target_category": relevant_category,
            "AP": ap,
            "Precision@10": p10,
            f"nDCG@{ndcg_k}": ndcg
        })

    summary = {
        "MAP": float(np.mean(ap_scores)),
        "Precision@10": float(np.mean(p10_scores)),
        f"nDCG@{ndcg_k}": float(np.mean(ndcg_scores))
    }

    return summary, pd.DataFrame(per_query_results)

In [22]:
import time

k1 = 1.5
b = 0.75
bm25 = BM25(corpus=processed_corpus, k1=k1, b=b)
start_time = time.time()

summary_test, per_query_test = evaluate_bm25_model(
    bm25=bm25,
    query_df=query,
    query_prepro_list=query_prepro,
    ndcg_k=10,
    max_queries=5
)

end_time = time.time()
total_time = end_time - start_time

print("\nFinal Summary")
print({"k1": k1, "b": b, **summary_test})

print("\nPer-query Results:")
print(per_query_test)

print(f"\nTotal time taken: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")


Final Summary
{'k1': 1.5, 'b': 0.75, 'MAP': 0.5976855742948282, 'Precision@10': 0.9199999999999999, 'nDCG@10': 0.9436725594537247}

Per-query Results:
                                query target_category        AP  Precision@10  \
0         stock market interest rates        business  0.581853           1.0   
1     company profits economic growth        business  0.749585           0.9   
2           oil prices global economy        business  0.635847           1.0   
3    bank inflation consumer spending        business  0.502794           1.0   
4  business merger corporate earnings        business  0.518349           0.7   

    nDCG@10  
0  1.000000  
1  0.921602  
2  1.000000  
3  1.000000  
4  0.796761  

Total time taken: 498.78 seconds (8.31 minutes)


In [23]:
import time

bm25 = BM25(corpus=processed_corpus, k1=k1, b=b)

start_time = time.time()

summary, per_query_results = evaluate_bm25_model(
    bm25=bm25,
    query_df=query,
    query_prepro_list=query_prepro,
    ndcg_k=10
)

end_time = time.time()
total_time = end_time - start_time

print("\nFinal Summary")
print({"k1": k1, "b": b, **summary})

print("\nPer-query Results:")
print(per_query_results)

print(f"\nTotal time taken: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")


Final Summary
{'k1': 1.5, 'b': 0.75, 'MAP': 0.592978619859777, 'Precision@10': 0.9319999999999999, 'nDCG@10': 0.9442943305636432}

Per-query Results:
                                   query target_category        AP  \
0            stock market interest rates        business  0.581853   
1        company profits economic growth        business  0.749585   
2              oil prices global economy        business  0.635847   
3       bank inflation consumer spending        business  0.502794   
4     business merger corporate earnings        business  0.518349   
5                 film awards best actor   entertainment  0.775390   
6                pop music chart success   entertainment  0.592782   
7         television celebrity interview   entertainment  0.400435   
8              cinema release box office   entertainment  0.443991   
9            album launch music industry   entertainment  0.456763   
10          government election campaign        politics  0.728065   
11       

In [ ]:
# change code to rank results by relevance instead of category match
# create code to store results in a csv dataframe for each query with columns: query, target_category, AP, Precision@10, nDCG@10 and show the top 5 queries & category with the highest AP scores
# create code to state the best performing k1 and b values based on the evaluation results
